In [1]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader

/Users/artyom-olenev/Projects/news_analyzer/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3.2 Предобработка текстового корпуса

Перед подачей в языковую модель тексты очищаются от пустых значений и артефактов форматирования. Явная лемматизация не применяется: субword-токенизатор обрабатывает морфологию самостоятельно. Длина входа ограничена 256 токенами (покрывает > 90% новостей).

Числовые оценки GPT-4o (диапазон [−1; +1]) конвертируются в трёхклассовые метки по порогам ±0,1: POSITIVE (≥ 0,1), NEGATIVE (≤ −0,1), NEUTRAL (иначе). Разбивка 85 / 15 % с фиксированным зерном.

In [2]:
news  = pd.read_parquet(Path("news_descriptions/news_collection_old.parquet"))
gpt4o = pd.read_json(Path("news_descriptions/news_descriptions_GPT4o.json")).transpose()

df = pd.concat([news, gpt4o], axis=1)[:15000]
df = df[df["body"].astype(str) != '""'].reset_index(drop=True)
df["sentiment_score"] = pd.to_numeric(df["sentiment_score"], errors="coerce")
df = df.dropna(subset=["sentiment_score"]).reset_index(drop=True)
df["label"] = df["sentiment_score"].apply(
    lambda s: 2 if s >= 0.1 else (0 if s <= -0.1 else 1)
)

print(df["label"].map({0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}).value_counts())
print(f"Итого: {len(df)}")

label
POSITIVE    7351
NEGATIVE    3990
NEUTRAL     3431
Name: count, dtype: int64
Итого: 14772


## 3.3 Архитектура модели

Базовая модель: `cointegrated/rubert-tiny2` — компактная (Tiny) дистиллированная версия RuBERT, предобученная на русскоязычных текстах. Суффикс «tiny» означает существенно сокращённое по сравнению с полным RuBERT число слоёв и размерность скрытого состояния, что обеспечивает быстрый инференс при сохранении достаточного качества для доменной адаптации. Над энкодером добавляется линейная голова Linear(312, 3), принимающая вектор токена [CLS]. Выход — три класса: NEGATIVE (−1), NEUTRAL (0), POSITIVE (+1).

| Параметр | Значение |
|---|---|
| Слоёв энкодера | 2 |
| Размерность скрытого состояния | 312 |
| Голов самовнимания | 12 |
| Словарь токенизатора | 83 553 |
| Максимальная длина входа | 256 токенов |
| Обучаемых параметров | ~29 млн |
| Функция потерь | кросс-энтропия |

In [3]:
MODEL_NAME = "cointegrated/rubert-tiny2"
MAX_LEN    = 256
ID2LABEL   = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}
LABEL2ID   = {v: k for k, v in ID2LABEL.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

rng   = np.random.default_rng(42)
idx   = rng.permutation(len(df))
n_val = int(0.15 * len(df))

train_df = df.iloc[idx[n_val:]].reset_index(drop=True)
val_df   = df.iloc[idx[:n_val]].reset_index(drop=True)
print(f"Train: {len(train_df)}  Val: {len(val_df)}")

Train: 12557  Val: 2215


## 3.4 Дообучение модели

Полное дообучение всех весов (full fine-tuning). Оптимизатор AdamW, lr = 2×10⁻⁵, weight_decay = 0,01, batch = 32, 3 эпохи.

| Гиперпараметр | Значение |
|---|---|
| Обучающая выборка | ~12 550 |
| Валидационная выборка | ~2 220 |
| Оптимизатор | AdamW |
| Скорость обучения | 2×10⁻⁵ |
| Регуляризация весов | 0,01 |
| Размер батча | 32 |
| Число эпох | 3 |

In [4]:
class FinSentDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc    = tokenizer(texts, truncation=True, padding=True,
                                max_length=MAX_LEN, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()}, self.labels[i]

def make_loader(df, batch_size, shuffle=False):
    return DataLoader(
        FinSentDataset(df["body"].fillna("").str.strip('"').tolist(),
                       df["label"].tolist()),
        batch_size=batch_size, shuffle=shuffle,
    )

train_loader = make_loader(train_df, batch_size=32, shuffle=True)
val_loader   = make_loader(val_df,   batch_size=64)

In [5]:
device = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))
print("Device:", device)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

for epoch in range(3):
    model.train()
    total_loss, steps = 0.0, 0
    for enc, labels in train_loader:
        enc    = {k: v.to(device) for k, v in enc.items()}
        labels = labels.to(device)
        optimizer.zero_grad()
        loss = model(**enc, labels=labels).loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        steps += 1
    print(f"Epoch {epoch + 1}/3  loss={total_loss / steps:.4f}")

Device: mps


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 9881.21it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the ch

Epoch 1/3  loss=0.8527
Epoch 2/3  loss=0.6666
Epoch 3/3  loss=0.5884


## 4.1 Качество классификатора

Оценка на валидационной выборке (~2 220 примеров, размеченных GPT-4o). Основная метрика — macro-F1 (дисбаланс классов: POSITIVE ≈ 50%, NEGATIVE ≈ 27%, NEUTRAL ≈ 23%). Базовая модель без дообучения даёт macro-F1 = 0,51.

In [6]:
model.eval()
preds = []
with torch.no_grad():
    for enc, _ in val_loader:
        enc = {k: v.to(device) for k, v in enc.items()}
        preds.extend(model(**enc).logits.argmax(dim=-1).cpu().tolist())

y_true   = val_df["label"].tolist()
f1_list  = []
for cls, name in ID2LABEL.items():
    tp   = sum(p == cls and t == cls for p, t in zip(preds, y_true))
    fp   = sum(p == cls and t != cls for p, t in zip(preds, y_true))
    fn   = sum(p != cls and t == cls for p, t in zip(preds, y_true))
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    f1_list.append(f1)
    print(f"{name:10s}  precision={prec:.3f}  recall={rec:.3f}  f1={f1:.3f}")

print(f"\nmacro-F1 = {sum(f1_list) / len(f1_list):.3f}")

NEGATIVE    precision=0.719  recall=0.765  f1=0.741
NEUTRAL     precision=0.641  recall=0.373  f1=0.472
POSITIVE    precision=0.755  recall=0.870  f1=0.809

macro-F1 = 0.674


In [7]:
SAVE_DIR = Path("models/rubert-financial-sentiment")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved → {SAVE_DIR}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 22.18it/s]

Saved → models/rubert-financial-sentiment
